# Session 2: Working with LLM Coding Agents

## Overview

This assignment explores how large language models function as coding agents within integrated development environments. You'll work with climate emissions data to understand both the capabilities and limitations of LLM-assisted data analysis, with particular attention to how execution environment shapes what's possible.

## Learning Objectives

Through structured exercises with the OWID CO2 dataset, you will:

- **Distinguish** between chat-based LLM interfaces and coding agents with local execution capabilities
- **Develop** fluency in specifying data operations through natural language while understanding the underlying computational patterns
- **Evaluate** the boundary between tasks coding agents handle autonomously versus those requiring human guidance
- **Assess** when programmatic approaches offer advantages over manual analysis methods

## The Execution Environment Question

The same foundation model—Claude Sonnet 4.5, GPT-4, or similar—exhibits markedly different capabilities depending on its execution context. A browser-based chat interface can generate syntactically correct code but lacks the infrastructure to execute, test, and refine it. The model generates a response and moves on.

In contrast, a coding agent operating within VS Code has access to a Python interpreter, your installed packages, the filesystem, and terminal. This enables an iterative cycle: execute code, parse errors or unexpected output, modify the approach, and re-execute. The model maintains context across this loop, accumulating information about what works in your specific environment.

This architectural difference—not model sophistication—determines whether you can request "analyze emissions trends by region" and receive working results, or whether you must manually execute generated code, diagnose failures, and prompt for corrections.

## Working with OWID Climate Data

Our World in Data maintains a comprehensive CO2 emissions dataset with country-level historical data, fuel source breakdowns, and economic indicators. The dataset's accessible format (direct CSV URL, documented schema, standard tabular structure) makes it appropriate for examining how coding agents handle common analytical workflows: filtering, aggregation, reshaping, and visualization.

Your role is to specify analyses in natural language and observe how the coding agent approaches each task. Note what it handles autonomously, where it requires clarification, what libraries it selects, and how it recovers from errors.

---
## Part 0: Setup

Import the necessary libraries for data manipulation and visualization. Request this from your coding agent and observe library selection.

In [47]:
# Setup - import libraries


---
## Part 1: Loading and Initial Exploration

### Task 1.1: Load and Explore

Load the OWID CO2 dataset: `https://github.com/owid/co2-data/raw/master/owid-co2-data.csv`

Request from your coding agent:
1. Load the data using appropriate library
2. Report dimensions (rows × columns)
3. Identify temporal coverage
4. Display sample rows
5. List emissions-related columns

In [1]:
# Task 1.1 - Your code here
import pandas as pd

URL = "https://github.com/owid/co2-data/raw/master/owid-co2-data.csv"

# 1. Load the data directly from the URL
df = pd.read_csv(URL)

# 2. Report dimensions (rows × columns)
print(f"Dimensions: {df.shape[0]} rows × {df.shape[1]} columns")

# 3. Identify temporal coverage
print(f"Temporal coverage: {df['year'].min()} – {df['year'].max()}")

# 4. Display sample rows
display(df.head())

# 5. List emissions-related columns
emissions_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in ["co2", "ghg", "methane", "nitrous"])
]
print(f"Emissions-related columns ({len(emissions_cols)}):")
for c in emissions_cols:
    print(f"  - {c}")

Dimensions: 50411 rows × 79 columns
Temporal coverage: 1750 – 2024


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Emissions-related columns (69):
  - cement_co2
  - cement_co2_per_capita
  - co2
  - co2_growth_abs
  - co2_growth_prct
  - co2_including_luc
  - co2_including_luc_growth_abs
  - co2_including_luc_growth_prct
  - co2_including_luc_per_capita
  - co2_including_luc_per_gdp
  - co2_including_luc_per_unit_energy
  - co2_per_capita
  - co2_per_gdp
  - co2_per_unit_energy
  - coal_co2
  - coal_co2_per_capita
  - consumption_co2
  - consumption_co2_per_capita
  - consumption_co2_per_gdp
  - cumulative_cement_co2
  - cumulative_co2
  - cumulative_co2_including_luc
  - cumulative_coal_co2
  - cumulative_flaring_co2
  - cumulative_gas_co2
  - cumulative_luc_co2
  - cumulative_oil_co2
  - cumulative_other_co2
  - flaring_co2
  - flaring_co2_per_capita
  - gas_co2
  - gas_co2_per_capita
  - ghg_excluding_lucf_per_capita
  - ghg_per_capita
  - land_use_change_co2
  - land_use_change_co2_per_capita
  - methane
  - methane_per_capita
  - nitrous_oxide
  - nitrous_oxide_per_capita
  - oil_co2
  - oil_

### Task 1.2: Filtering and Ranking

Identify the top 5 CO2 emitting countries (not aggregate regions) in 2022. test

**Note:** Dataset includes aggregate entities (World, continents, income groups). The coding agent needs to distinguish actual countries - observe whether it infers this from ISO codes or requires explicit guidance.

In [2]:
# Task 1.2 - Your code here
import pandas as pd

# Note: 'iso_code' is NaN for aggregates (World, continents, income groups).
# Filtering on it is the reliable way to keep only actual countries.
countries_2022 = (
    df[(df["year"] == 2022) & (df["iso_code"].notna())]
    .dropna(subset=["co2"])          # keep rows with an actual emissions value
    .sort_values("co2", ascending=False)
)

top5 = countries_2022.head(5)[["country", "iso_code", "co2"]].reset_index(drop=True)
print("Top 5 CO2 emitting countries, 2022:\n")
display(top5)

# Sanity check: what got excluded by the iso_code filter?
excluded = df[(df["year"] == 2022)]["country"].nunique() - countries_2022["country"].nunique()
print(f"(Excluded {excluded} aggregate entities via the iso_code filter)")

Top 5 CO2 emitting countries, 2022:



,country,iso_code,co2
0,China,CHN,11711.808
1,United States,USA,5055.403
2,India,IND,2831.132
3,Russia,RUS,1675.461
4,Japan,JPN,1029.645


(Excluded 39 aggregate entities via the iso_code filter)


In [4]:
# Task 1.3 - CO2 emissions trajectories for top 5 emitters (1990-2022)
import altair as alt

# Reuse the top-5 list computed in Task 1.2
top5_names = top5["country"].tolist()

# Filter the full frame to those countries, the window, and non-missing values
trajectory = (
    df[(df["country"].isin(top5_names)) & (df["year"] >= 1990) & (df["year"] <= 2022)]
    .dropna(subset=["co2"])
    .sort_values("year")
)

chart = (
    alt.Chart(trajectory)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d")),
        y=alt.Y("co2:Q", title="CO₂ emissions (million tonnes)"),
        color=alt.Color("country:N", title="Country"),
        tooltip=["country", "year", alt.Tooltip("co2:Q", title="CO₂ (Mt)", format=",.0f")],
    )
    .properties(
        title=alt.Title("CO₂ Emissions of the Top 5 Emitters, 1990–2022", anchor="start"),
        width=700,
        height=400,
    )
)

chart

alt.Chart(...)

### Task 1.3: Time Series Visualization

Create a line chart showing CO2 emissions trajectories for the top 5 emitters (1990-2022).

**Expected observation:** China overtaking US around 2005-2006.

**What to notice:** Does the agent select appropriate visualization library? Handle missing data? Create informative labels?

### Task 1.4: Compositional Analysis

For the top 5 countries in 2022, visualize emissions breakdown by source:
- Coal (`coal_co2`)
- Oil (`oil_co2`)
- Gas (`gas_co2`)  
- Cement (`cement_co2`)

Use a stacked bar chart or equivalent compositional visualization.

**Data reshaping challenge:** This requires pivoting from wide format (separate columns per source) to long format (source as variable). The coding agent should handle this transformation autonomously.

In [4]:
# Task 1.4 - Emissions breakdown by source for top 5 emitters (2022)
import pandas as pd
import altair as alt

# --- Self-contained data load: only re-reads the CSV if df is missing ---
URL = "https://github.com/owid/co2-data/raw/master/owid-co2-data.csv"
if "df" not in globals():
    df = pd.read_csv(URL)

source_cols = ["coal_co2", "oil_co2", "gas_co2", "cement_co2"]

# --- Recompute top 5 emitters (2022) from df: no dependence on Task 1.2 ---
top5_names = (
    df[(df["year"] == 2022) & (df["iso_code"].notna())]
    .dropna(subset=["co2"])
    .sort_values("co2", ascending=False)
    .head(5)["country"]
    .tolist()
)
print("Top 5:", top5_names)

# --- 1. Slice the wide table: top 5 countries, 2022, only the source columns ---
wide = (
    df[(df["country"].isin(top5_names)) & (df["year"] == 2022)]
    [["country"] + source_cols]
    .reset_index(drop=True)
)

# --- 2. Pivot wide -> long: four columns become (source, value) pairs ---
long = wide.melt(
    id_vars="country",
    value_vars=source_cols,
    var_name="source",
    value_name="co2_mt",
).dropna(subset=["co2_mt"])

# --- 3. Nice labels + fixed stacking order (bottom -> top) ---
source_labels = {
    "coal_co2": "Coal",
    "oil_co2": "Oil",
    "gas_co2": "Gas",
    "cement_co2": "Cement",
}
long["source"] = long["source"].map(source_labels)

order = ["Coal", "Oil", "Gas", "Cement"]  # stacking order, matches Altair's color domain
long["source"] = pd.Categorical(long["source"], categories=order, ordered=True)
long = long.sort_values(["country", "source"]).reset_index(drop=True)

# --- 4. Compute segment midpoints for centered data labels ---
long["cum_sum"] = long.groupby("country")["co2_mt"].cumsum()
long["mid_y"] = long["cum_sum"] - long["co2_mt"] / 2

# --- 5. Stacked bar chart with legend ---
bars = (
    alt.Chart(long)
    .mark_bar()
    .encode(
        x=alt.X("country:N", title="Country", sort=top5_names),
        y=alt.Y("co2_mt:Q", title="CO₂ emissions (million tonnes)"),
        color=alt.Color(
            "source:N",
            title="Fuel source",
            legend=alt.Legend(title="Fuel source"),
            scale=alt.Scale(scheme="category10", domain=order),
        ),
        tooltip=[
            "country:N",
            "source:N",
            alt.Tooltip("co2_mt:Q", title="CO₂ (Mt)", format=",.0f"),
        ],
    )
)

# --- 6. Data labels, white bold, centered in each segment ---
labels = (
    alt.Chart(long)
    .mark_text(fontSize=10, fontWeight="bold", color="white",
               baseline="middle", align="center")
    .encode(
        x=alt.X("country:N", sort=top5_names),
        y=alt.Y("mid_y:Q", axis=None),
        text=alt.Text("co2_mt:Q", format=",.0f"),
    )
)

chart = (
    (bars + labels)
    .properties(
        title=alt.Title(
            "CO₂ Emissions by Source (Mt) — Top 5 Emitters, 2022",
            anchor="start",
        ),
        width=600,
        height=400,
    )
)

chart

Top 5: ['China', 'United States', 'India', 'Russia', 'Japan']


alt.LayerChart(...)

---
## Part 2: Advanced Operations

### Task 2.1: Aggregation by Grouping Variable

Calculate average per capita CO2 emissions by continent for 2022. Return results sorted descending.

**Key operation:** This requires `groupby()` with aggregation. The coding agent should:
1. Identify the continent variable in the schema
2. Filter to 2022 and valid per capita values
3. Compute mean by group
4. Handle missing data appropriately

In [7]:
# Task 2.1 - Your code here
# Task 2.1 - Average per capita CO2 emissions by continent (2022)
import pandas as pd

# --- Self-contained data load ---
URL = "https://github.com/owid/co2-data/raw/master/owid-co2-data.csv"
if "df" not in globals():
    df = pd.read_csv(URL)

# There is no continent column; continents live as values in 'country'.
continents = ["Africa", "Asia", "Europe", "North America", "Oceania", "South America"]

# 1. Filter to 2022, take only continent rows, drop missing per-capita values
cont = (
    df[(df["year"] == 2022) & (df["country"].isin(continents))]
    .dropna(subset=["co2_per_capita"])
    [["country", "co2_per_capita"]]
)

# 2. Aggregate by continent, descending
by_continent = (
    cont.groupby("country", as_index=False)["co2_per_capita"]
    .mean()                                      # groupby aggregation
    .sort_values("co2_per_capita", ascending=False)
    .reset_index(drop=True)
)

by_continent


,country,co2_per_capita
0,North America,10.390
1,Oceania,9.679
2,Europe,6.813
3,Asia,4.685
4,South America,2.557
5,Africa,1.010


### Task 2.2: Percentage Change Calculation

Identify countries with:
1. Five largest emission increases (2010→2022, percentage)
2. Five largest emission decreases (2010→2022, percentage)

Visualize both groups for comparison.

**Complexity:** Requires joining data across years, computing percentage change, handling countries with missing baseline or endpoint data. Observe error handling strategies.

In [6]:
# Task 2.2 - Biggest % emission changes (2010 -> 2022)
import numpy as np
import pandas as pd
import altair as alt

# --- Self-contained data load ---
URL = "https://github.com/owid/co2-data/raw/master/owid-co2-data.csv"
if "df" not in globals():
    df = pd.read_csv(URL)

# --- 1. Keep only real countries (exclude World, continents, income groups) ---
countries = df[df["iso_code"].notna()].copy()

# --- 2. Pull the two years into wide columns (join across years) ---
y2010 = countries.loc[countries["year"] == 2010, ["country", "co2"]].rename(
    columns={"co2": "co2_2010"}
)
y2022 = countries.loc[countries["year"] == 2022, ["country", "co2"]].rename(
    columns={"co2": "co2_2022"}
)

merged = y2010.merge(y2022, on="country", how="inner")

# --- 3. Handle missing data: keep only rows with BOTH years present ---
merged = merged.dropna(subset=["co2_2010", "co2_2022"])
print(f"Countries with valid 2010 AND 2022 emissions: {len(merged)}")

# --- 4. Compute percentage change, guarding against zero/NaN baselines ---
merged["pct_change"] = (
    (merged["co2_2022"] - merged["co2_2010"]) / merged["co2_2010"]
) * 100

valid = merged[np.isfinite(merged["pct_change"])].copy()
# (baseline of exactly 0 with positive endpoint would yield inf -> filtered above)
print(f"Valid percentage changes: {len(valid)}")

# --- 5. Rank both directions ---
increases = valid.nlargest(5, "pct_change")
decreases = valid.nsmallest(5, "pct_change")

print("\nLargest INCREASES 2010->2022 (%):")
display(increases[["country", "co2_2010", "co2_2022", "pct_change"]])

print("\nLargest DECREASES 2010->2022 (%):")
display(decreases[["country", "co2_2010", "co2_2022", "pct_change"]])

# --- 6. Visualize both groups for comparison ---
both = pd.concat([
    increases.assign(group="Increase"),
    decreases.assign(group="Decrease"),
])

chart = (
    alt.Chart(both)
    .mark_bar()
    .encode(
        x=alt.X("country:N", title="Country",
                sort=alt.SortField("pct_change", order="descending")),
        y=alt.Y("pct_change:Q", title="CO₂ change 2010→2022 (%)"),
        color=alt.Color("group:N", title="Direction",
                        scale=alt.Scale(domain=["Increase", "Decrease"],
                                        range=["#e45756", "#54a24b"])),
        tooltip=[
            "country:N",
            alt.Tooltip("co2_2010:Q", title="2010 (Mt)", format=",.0f"),
            alt.Tooltip("co2_2022:Q", title="2022 (Mt)", format=",.0f"),
            alt.Tooltip("pct_change:Q", title="Δ %", format=".1f"),
        ],
    )
    .properties(
        title=alt.Title(
            "Top 5 % Increases vs Top 5 % Decreases in CO₂ (2010→2022)",
            anchor="start",
        ),
        width=650,
        height=400,
    )
)

chart

Countries with valid 2010 AND 2022 emissions: 215
Valid percentage changes: 213

Largest INCREASES 2010->2022 (%):


,country,co2_2010,co2_2022,pct_change
105,Laos,3.001,22.549,651.382872
192,Tajikistan,2.535,10.521,315.029586
34,Cambodia,5.067,20.865,311.782120
136,Nepal,4.828,17.609,264.726595
132,Mozambique,2.638,9.296,252.388173



Largest DECREASES 2010->2022 (%):


,country,co2_2010,co2_2022,pct_change
10,Aruba,2.506,0.888,-64.565044
215,Yemen,25.818,10.404,-59.702533
130,Montserrat,0.059,0.026,-55.932203
204,Ukraine,294.366,143.040,-51.407432
190,Syria,62.057,31.423,-49.364294


alt.Chart(...)

In [3]:
# Task 1.4 - Emissions breakdown by source for top 5 emitters (2022)
import pandas as pd
import altair as alt

# --- Self-contained data load: only re-reads the CSV if df is missing ---
URL = "https://github.com/owid/co2-data/raw/master/owid-co2-data.csv"
if "df" not in globals():
    df = pd.read_csv(URL)

source_cols = ["coal_co2", "oil_co2", "gas_co2", "cement_co2"]

# --- Recompute top 5 emitters (2022) from df: no dependence on Task 1.2 ---
top5_names = (
    df[(df["year"] == 2022) & (df["iso_code"].notna())]
    .dropna(subset=["co2"])
    .sort_values("co2", ascending=False)
    .head(5)["country"]
    .tolist()
)
print("Top 5:", top5_names)

# --- 1. Slice the wide table: top 5 countries, 2022, only the source columns ---
wide = (
    df[(df["country"].isin(top5_names)) & (df["year"] == 2022)]
    [["country"] + source_cols]
    .reset_index(drop=True)
)

# --- 2. Pivot wide -> long: four columns become (source, value) pairs ---
long = wide.melt(
    id_vars="country",
    value_vars=source_cols,
    var_name="source",
    value_name="co2_mt",
).dropna(subset=["co2_mt"])

# --- 3. Nice labels for the legend ---
source_labels = {
    "coal_co2": "Coal",
    "oil_co2": "Oil",
    "gas_co2": "Gas",
    "cement_co2": "Cement",
}
long["source"] = long["source"].map(source_labels)

# --- 4. Stacked bar chart ---
chart = (
    alt.Chart(long)
    .mark_bar()
    .encode(
        x=alt.X("country:N", title="Country", sort=top5_names),
        y=alt.Y("sum(co2_mt):Q", title="CO₂ emissions (million tonnes)"),
        color=alt.Color("source:N", title="Fuel source",
                        scale=alt.Scale(scheme="category10")),
        tooltip=[
            "country:N",
            "source:N",
            alt.Tooltip("co2_mt:Q", title="CO₂ (Mt)", format=",.0f"),
        ],
    )
    .properties(
        title=alt.Title(
            "CO₂ Emissions by Source — Top 5 Emitters, 2022",
            anchor="start",
        ),
        width=600,
        height=400,
    )
)

chart

Top 5: ['China', 'United States', 'India', 'Russia', 'Japan']


alt.Chart(...)

### Task 2.3: Faceted Visualization

Create small multiples showing per capita CO2 trends (1990-2022) with one panel per continent.

**Visualization technique:** Faceting/small multiples allow pattern comparison across categories. The coding agent should select appropriate library (Altair, matplotlib with subplots, etc.) and configure layout.

In [45]:
# Task 2.3 - Your code here



---
## Part 3: Independent Analysis

### Task 3.1: Design and Execute Analysis

Select ONE question and complete the analysis:

**Option A:** Emissions Intensity Improvements
- Identify countries with largest reductions in CO2 per unit GDP (2000-2022)
- Visualize as efficiency gains

**Option B:** Coal Transition Analysis
- Find countries that significantly reduced coal's share of total emissions (2010-2022)
- Show before/after fuel composition

**Option C:** Population-Emissions Relationship
- Analyze correlation between population growth and emissions growth
- Create annotated scatter plot highlighting outliers

**Deliverables:**
- Working code that executes without errors
- At least one publication-quality visualization
- Interpretation of findings and limitations

### My Analysis:

**Question I'm investigating:**

[Describe your chosen question here]

In [46]:
# Your analysis code here



### Analysis Summary

**Research question:**

**Methodology:**

**Key findings:**

**Limitations and caveats:**

---
## Part 4: Critical Reflection

### Evaluating Coding Agent Performance

Analyze your experience across the preceding exercises:

**1. Autonomous Completion Rate**  
What proportion of tasks executed correctly on first attempt? Where did the agent require clarification, correction, or multiple iterations?

**2. Error Recovery Patterns**  
When code failed, document the agent's diagnostic approach. Did it parse error messages effectively? Make appropriate modifications? Or require explicit guidance to identify the problem?

**3. Technical Choices**  
Examine the agent's library selections (ibis, pandas, polars) and data manipulation strategies. Were choices appropriate for the data scale and operation complexity? Could you identify more efficient approaches?

**4. Edge Case Handling**  
How did the agent address missing data, type inconsistencies, or ambiguous specifications? What assumptions did it make, and were they reasonable?

**5. Conceptual Transfer**  
You've worked with groupby aggregation, wide-to-long reshaping, faceted visualization, and percentage change calculations. Can you now explain these patterns to a colleague and recognize when to apply them, independent of specific syntax?

**6. Execution Environment Dependency**  
Which tasks could a browser-based LLM complete versus those requiring local execution? Where precisely does the boundary lie?

**7. Workflow Integration**  
How does this approach to exploratory analysis compare to your current methods? Where do you see coding agents adding value in professional contexts? Where do they introduce friction or uncertainty?

**Your analysis:**

1. 

2. 

3. 

4. 

5. 

6. 

7. 

---
## Submission Guidelines

**Required Components:**
1. Executed notebook with all code cells producing correct output
2. Completed reflection responses in Part 4
3. Original analysis from Part 3 with supporting visualization

**Evaluation Framework:**

*Technical Execution (30%)*  
Code quality, error handling, appropriate method selection, efficient data operations

*Conceptual Understanding (30%)*  
Demonstrated grasp of data manipulation patterns and coding agent capabilities through reflection responses

*Visualization Design (20%)*  
Clear, accurate, publication-ready graphics with appropriate encodings

*Analytical Depth (20%)*  
Part 3 investigation shows thoughtful problem formulation, appropriate scope, and clear interpretation

The assessment focuses on your understanding of how coding agents operate within execution environments and when to rely on versus scrutinize their output—not on independent Python proficiency.